# 02. Object detection by hand — preprocessing · reading the output · cleaning up overlapping boxes, with no library

> In the edge-lab server, the `easydetect` library does this whole process for you. Here we **let OpenVINO do only the model math, and write everything else ourselves**.
> We follow by hand what an object detection model really outputs, and how to cut that down to "a few boxes".

The model is **ed-m** — the very same model that edge-lab's [Objects] screen uses (easydetect, Apache-2.0, 80 COCO classes).
OpenVINO reads the same `.onnx` file that goes into Pibo · PiBrain directly.

### What you learn in this session
1. Building the input — stretch to 640 x 640 · RGB · 0~1 · NCHW
2. What the outputs `boxes [1, 300, 4]` · `scores [1, 300, 80]` mean — what is 300, and why is it so few?
3. Score filtering → corner coordinates → back to the original photo size
4. **IoU** and removing overlapping boxes (**NMS**) — write it yourself, and see why ed hardly needs it
5. Timing each step
6. (Bonus) 17 body joints — the **2-step** method: crop the person box and feed it to a second model

### The whole flow
```
photo ─stretch to 640x640─▶ ─RGB·÷255·NCHW─▶ [1,3,640,640]
     ─infer─▶ boxes [1,300,4] (center x, y, width, height · 0~1) + scores [1,300,80] (0~1)
     ─score filter─▶ a few ─corners × original size─▶ ─clean up overlaps (IoU 0.7)─▶ boxes on the original photo
```

## 1. Setup — the model and class names

In [ ]:
import os
import json
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "../..")).resolve()       # program folder (lab.bat tells us)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # model folder
core = ov.Core()
print("Model folder:", MODELS)
print("Devices:", core.available_devices)

def pick(*prefer):
    """Pick a device this PC has, trying them in the order given."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

In [ ]:
ED = MODELS / "easydetect" / "ed-m" / "ed-m.onnx"        # downloaded by setup from Hugging Face (leeyunjai/easydetect)
model = core.read_model(ED)
for i in model.inputs:
    print("Input:", i.any_name, i.partial_shape)
for o in model.outputs:
    print("Output:", o.any_name, o.partial_shape)

# Class names — ed-m.names.json next to the model (80 COCO classes)
NAMES = {int(k): v for k, v in json.loads((ED.parent / "ed-m.names.json").read_text(encoding="utf-8")).items()}
print(len(NAMES), "classes:", list(NAMES.values())[:8], "...")

**ed-m** — a real-time object detection model from the DETR (DEtection TRansformer) family. `m` means medium size (about 20 million numbers).
It was trained on COCO, a public set of photos. The input is **fixed at 640 x 640**, and there are **two** outputs (`boxes`, `scores`).

## 2. Photo

In [ ]:
PHOTO = "sample.jpg"          # photo to use when there is no webcam (same folder as this notebook)

def show(bgr, width=720):
    """Show an OpenCV photo (BGR) inside the notebook."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """Grab one picture from the webcam. If there is no webcam, use the PHOTO file."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # some webcams give dark pictures for the first few frames
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"No webcam and no {PHOTO} either — put a photo file name in PHOTO"
        print("No webcam, so using the file:", PHOTO)
    return frame

frame = grab()
print("Photo size (height, width, channels):", frame.shape)
show(frame)

## 3. Building the input — 640 x 640 · RGB · 0~1 · NCHW
The input is square, but a webcam photo is 4:3. There are two ways to handle this.

| Method | What it does | Used by |
|---|---|---|
| **Just stretch** | `cv2.resize(photo, (640, 640))` — stretches the width and height separately | **ed** learned this way |
| letterbox | Shrinks while keeping the ratio, and fills the leftover space with gray | YOLO family |

**Feed the model the same way it learned.** ed learned from stretched photos, so we stretch. People may look a bit thin, but the model is used to photos like that.
When you map the boxes back later, the **width scale (`w`) and height scale (`h`) are separate** — but ed gives boxes as 0~1 ratios, so you just multiply by the original width · height.

ed learned from **RGB** photos divided down to **0~1** (the face model in notebook 01 used BGR · 0~255 — every model is different!).
`np.ascontiguousarray` puts an array with swapped axes back in memory in order (the inference engine reads it faster).

In [ ]:
SIZE = 640

def to_blob(bgr, size=SIZE):
    x = cv2.resize(bgr, (size, size))                 # just stretch (the way ed learned)
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)            # it learned with RGB
    x = x.astype(np.float32) / 255.0                  # 0~255 → 0~1
    x = x.transpose(2, 0, 1)[None]                    # HWC → NCHW
    return np.ascontiguousarray(x)

blob = to_blob(frame)
print(blob.shape, blob.dtype, "value range:", blob.min(), "~", blob.max())
show(cv2.resize(frame, (SIZE, SIZE)), 400)            # the photo the model sees — a bit stretched

## 4. Inference and reading the output — `boxes [1, 300, 4]` · `scores [1, 300, 80]`

**What is 300?** ed looks at the photo and fills **"300 object-candidate slots"** (these are called queries). For each slot:
- `boxes` : `[center x, center y, width, height]` — **ratios of the photo size (0~1)**. Not pixels.
- `scores` : a score (0~1) for each of the 80 classes. The biggest one gives that slot's class · confidence.

If a photo has only 3 objects, only 3 of the 300 slots get high scores, and the rest stay low, meaning "nothing".
**ed was trained to use only one slot per object** — so it rarely gives several overlapping boxes for the same object (section 6).

In [ ]:
DEVICE = pick("GPU", "NPU", "CPU")
compiled = core.compile_model(model, DEVICE)
res = compiled(blob)
boxes_n = res[compiled.output("boxes")][0]            # [300, 4] — center x, y, width, height (0~1)
scores_all = res[compiled.output("scores")][0]        # [300, 80]
print(DEVICE, "boxes", boxes_n.shape, "scores", scores_all.shape, "score range", round(float(scores_all.min()), 3), "~", round(float(scores_all.max()), 3))

cls = scores_all.argmax(1)                            # the top class for each slot
conf = scores_all.max(1)                              # its score
print("Slots with a score over 0.5:", int((conf > 0.5).sum()), "/ 300")

top = np.argsort(-conf)[:5]                           # the five highest scores
for i in top:
    cx, cy, w, h = boxes_n[i]
    print(f"  slot {i:3d}: {NAMES[int(cls[i])]:12s} score {conf[i]:.2f}  center({cx:.2f}, {cy:.2f}) size {w:.2f}x{h:.2f}")

## 5. Score filtering → corner coordinates → back to the original photo size
- **Score filtering**: ed gives 0.3~0.5 even to slots it is not sure about (YOLO gives below 0.25). So we set the threshold at **0.5**.
- **Corner coordinates**: drawing · IoU use the corners `(x1, y1, x2, y2)`. `x1 = cx − w/2`, `x2 = cx + w/2` …
- **Back to the original size**: the values are 0~1 ratios, so **multiply x by the original width and y by the original height**. We stretched the photo, so the width and height scales are different, but because these are ratios, one multiplication is all you need.

$$x_{\text{original}} = x_{0\sim1} \times \text{width}, \qquad y_{\text{original}} = y_{0\sim1} \times \text{height}$$

In [ ]:
def to_xyxy(b, shape):
    h, w = shape[:2]
    xyxy = np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,     # center · size → corners
                     b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], 1)
    xyxy *= np.array([w, h, w, h], np.float32)                         # 0~1 → original photo pixels
    xyxy[:, [0, 2]] = xyxy[:, [0, 2]].clip(0, w)
    xyxy[:, [1, 3]] = xyxy[:, [1, 3]].clip(0, h)
    return xyxy

m = conf > 0.5
xyxy0, cls0, conf0 = to_xyxy(boxes_n[m], frame.shape), cls[m], conf[m]
order = conf0.argsort()[::-1]                                          # highest score first
xyxy0, cls0, conf0 = xyxy0[order], cls0[order], conf0[order]
for (x1, y1, x2, y2), c, s in zip(xyxy0.astype(int), cls0, conf0):
    print(f"{NAMES[int(c)]:14s} {s:.2f}  ({x1}, {y1}) - ({x2}, {y2})")

## 6. IoU — how much do two boxes overlap?
Models like YOLO find the same person from many slots at once, so you get dozens of almost identical boxes. To cut them down to one, you need to measure "how much they overlap" as a number.

$$\text{IoU} = \frac{\text{overlap area}}{\text{area of both boxes combined}} = \frac{A \cap B}{A + B - A \cap B}$$

0 means no overlap at all, and 1 means the exact same box. Let's calculate it by hand once.

In [ ]:
def iou(a, b):
    """a: [4] one box, b: [N, 4] many boxes (x1, y1, x2, y2)"""
    x1 = np.maximum(a[0], b[:, 0]); y1 = np.maximum(a[1], b[:, 1])     # top-left of the overlap
    x2 = np.minimum(a[2], b[:, 2]); y2 = np.minimum(a[3], b[:, 3])     # bottom-right of the overlap
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)      # no overlap gives a negative → 0
    area = lambda r: (r[..., 2] - r[..., 0]) * (r[..., 3] - r[..., 1])
    return inter / (area(a) + area(b) - inter + 1e-9)

A = np.array([0, 0, 100, 100.])
B = np.array([[50, 0, 150, 100.],     # half overlap: overlap area 50x100=5000, combined area 10000+10000-5000=15000 → 0.333
              [0, 0, 100, 100.],      # identical → 1
              [200, 200, 300, 300.]]) # no overlap → 0
print(np.round(iou(A, B), 3))

## 7. NMS — removing overlapping boxes (Non-Maximum Suppression)
**The algorithm**
1. Line the boxes up from the highest score to the lowest.
2. **Keep** the first one (the highest).
3. Any other box whose IoU with it is above the threshold is treated as "the same object" and **thrown away**.
4. Repeat from step 2 with the boxes that are left.

First, let's check it by eye with boxes we make ourselves.

In [ ]:
def nms(xyxy, scores, iou_th=0.45):
    order = scores.argsort()[::-1]                    # highest score first
    keep = []
    while len(order):
        i = order[0]
        keep.append(i)                                # keep the highest one
        rest = order[1:]
        order = rest[iou(xyxy[i], xyxy[rest]) < iou_th]  # throw away the ones that overlap it a lot
    return np.array(keep, dtype=int)

toy = np.array([[60, 60, 260, 300], [70, 50, 270, 290], [80, 70, 250, 310],   # the same object three times
                [330, 100, 520, 260], [340, 110, 530, 270]], float)           # a different object twice
toy_s = np.array([0.9, 0.8, 0.7, 0.85, 0.6])
kept = nms(toy, toy_s)
canvas = np.full((360, 600, 3), 255, np.uint8)
before, after = canvas.copy(), canvas.copy()
for b, s in zip(toy.astype(int), toy_s):
    cv2.rectangle(before, tuple(b[:2]), tuple(b[2:]), (0, 0, 255), 2)
for k in kept:
    b = toy[k].astype(int)
    cv2.rectangle(after, tuple(b[:2]), tuple(b[2:]), (0, 160, 0), 3)
    cv2.putText(after, f"{toy_s[k]:.2f}", (b[0], b[1] - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 160, 0), 2)
print("Kept boxes:", kept, "(left: 5 boxes before NMS, right: after NMS)")
show(np.hstack([before, after]), 900)

### Applying it to the real output — ed hardly needs it
ed learned to use one slot per object, so boxes that score over 0.5 mostly do not overlap.
Still, it sometimes finds **the same object as two classes** — a truck at 0.83 and a car at 0.57 in almost the same place (IoU 0.9 or more).
So edge-lab **removes the lower-scoring box when two boxes overlap by more than IoU 0.7, no matter the class** (the same rule as easydetect).
Unlike YOLO, where dozens of boxes get removed, this just tidies up one or two boxes that overlap a lot.

In [ ]:
def decode(boxes_n, scores_all, shape, conf_th=0.5, iou_th=0.7):
    cls, conf = scores_all.argmax(1), scores_all.max(1)
    m = conf > conf_th                                                 # ① score filtering
    xyxy, cls, conf = to_xyxy(boxes_n[m], shape), cls[m], conf[m]      # ② corner coordinates · original size
    keep = nms(xyxy, conf, iou_th)                                     # ③ tidy up only heavy overlaps (any class)
    return xyxy[keep], cls[keep], conf[keep], int(m.sum())

xyxy, cls_k, conf_k, n_pass = decode(boxes_n, scores_all, frame.shape)
print(f"300 candidates → {n_pass} pass the score → {len(xyxy)} after removing overlaps")
low = decode(boxes_n, scores_all, frame.shape, conf_th=0.25)
print(f"(with the threshold lowered to 0.25: {low[3]} pass → {len(low[0])} after cleanup — at low scores, the same object is sometimes found twice)")

view = frame.copy()
for (x1, y1, x2, y2), c, s in zip(xyxy.astype(int), cls_k, conf_k):
    color = tuple(int(v) for v in np.random.default_rng(int(c)).integers(60, 255, 3))   # the same color for each class
    cv2.rectangle(view, (x1, y1), (x2, y2), color, 2)
    cv2.putText(view, f"{NAMES[int(c)]} {s:.2f}", (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
show(view)

## 8. Where does the time go?
Fast inference is not the whole story. Preprocessing (resizing · changing colors · dividing) and postprocessing (filtering · removing overlaps) run on the **CPU**, whatever the device.
The faster inference gets on the GPU · NPU, the bigger the share of pre- and post-processing.

In [ ]:
def timed(n=20):
    t = {"pre": 0.0, "infer": 0.0, "post": 0.0}
    for _ in range(n):
        t0 = time.perf_counter(); x = to_blob(frame)
        t1 = time.perf_counter(); r = compiled(x)
        t2 = time.perf_counter(); decode(r[compiled.output("boxes")][0], r[compiled.output("scores")][0], frame.shape)
        t3 = time.perf_counter()
        t["pre"] += t1 - t0; t["infer"] += t2 - t1; t["post"] += t3 - t2
    return {k: v / n * 1000 for k, v in t.items()}

for k, v in timed().items():
    print(f"{k:6s} {v:7.2f} ms  {'█' * max(1, round(v))}")

## 9. (Bonus) 17 joints — feeding only the person into a second model (2 steps)
The pose model `pose-m` does not take the whole photo. It takes **a photo cropped to one person (height 256 x width 192)**. This is the same "find → crop → ask again" method as the face analysis in notebook 03.

**Cropping** (exactly the way the model learned)
1. Widen the person box to a height:width = 256:192 (4:3) shape, and give it **1.25x** extra room, because arms and legs stick out.
2. Move that area onto a 256 x 192 image — `cv2.warpAffine` (a 2x3 matrix) does the scaling and moving in one go.
3. The input is **RGB 0~255 as is** (no ÷255!) · NCHW. This model learned that way.

**Output** `x [N, 17, 384]` · `y [N, 17, 512]` — for each joint, **scores for 384 horizontal bins and 512 vertical bins**.
The bins split the 192-pixel width into half pixels (×2). **The highest bin ÷ 2** is the coordinate on the cropped photo, and you map it back to the original photo with the **inverse** of the step-2 matrix.
The confidence is turned into 0~1 as "highest bin score ÷ the score when perfectly sure".

In [ ]:
POSE = MODELS / "easydetect" / "pose" / "pose-m.onnx"
SKELETON = [(5, 7), (7, 9), (6, 8), (8, 10), (5, 6), (5, 11), (6, 12), (11, 12),
            (11, 13), (13, 15), (12, 14), (14, 16), (0, 1), (0, 2), (1, 3), (2, 4)]   # how the COCO joints connect
PH, PW, SPLIT, PAD = 256, 192, 2, 1.25

def crop_matrix(box):
    """Person box → a 2x3 matrix that moves it to 256x192 (widen to 4:3, plus 1.25x room)."""
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    w, h = max(x2 - x1, 1.0), max(y2 - y1, 1.0)
    if w > h * PW / PH: h = w * PH / PW          # make it 4:3 — stretch the side that is too short
    else:               w = h * PW / PH
    w, h = w * PAD, h * PAD
    sx, sy = PW / w, PH / h
    return np.array([[sx, 0, PW / 2 - sx * cx], [0, sy, PH / 2 - sy * cy]])

def peak(sigma, bins):
    """Highest bin score when perfectly sure (the training target shape) — the baseline for confidence"""
    d = np.arange(bins) - bins / 2
    g = np.exp(-d ** 2 / (2 * sigma ** 2))
    return g.max() / g.sum()

def softmax(z):
    z = z - z.max(-1, keepdims=True); e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

if not POSE.exists():
    print("No pose model, so skipping:", POSE)
else:
    pose = core.compile_model(POSE, DEVICE)
    people = xyxy[cls_k == 0]                                               # only the people found in section 7
    mats = [crop_matrix(b) for b in people]
    crops = [cv2.cvtColor(cv2.warpAffine(frame, mt, (PW, PH)), cv2.COLOR_BGR2RGB) for mt in mats]
    view = frame.copy()
    if crops:
        x_in = np.ascontiguousarray(np.stack(crops).transpose(0, 3, 1, 2).astype(np.float32))   # RGB 0~255 · NCHW
        r = pose(x_in)
        xl, yl = r[pose.output("x")], r[pose.output("y")]                    # [N,17,384] · [N,17,512]
        px, py = softmax(xl), softmax(yl)
        u, v = px.argmax(-1) / SPLIT, py.argmax(-1) / SPLIT                  # coordinates on the cropped photo
        kc = np.minimum(px.max(-1) / peak(4.9, PW * SPLIT), py.max(-1) / peak(5.66, PH * SPLIT)).clip(0, 1)
        for mt, uu, vv, cc in zip(mats, u, v, kc):
            sx, tx, sy, ty = mt[0, 0], mt[0, 2], mt[1, 1], mt[1, 2]
            pts = np.stack([(uu - tx) / sx, (vv - ty) / sy], 1)              # inverse transform — back to the original photo
            for a, c in SKELETON:
                if cc[a] > 0.3 and cc[c] > 0.3:                              # connect only when both are visible
                    cv2.line(view, tuple(pts[a].astype(int)), tuple(pts[c].astype(int)), (255, 160, 0), 3)
            for (x, y), k in zip(pts, cc):
                if k > 0.3:
                    cv2.circle(view, (int(x), int(y)), 4, (0, 0, 255), -1)
        show(np.hstack([cv2.cvtColor(c, cv2.COLOR_RGB2BGR) for c in crops[:4]]), 600)   # the person photos the model saw
    print("People:", len(crops))
    show(view)

## Common errors
| Symptom | Cause · fix |
|---|---|
| Nothing found | You forgot `÷255` or fed BGR as is — check section 3 |
| Boxes in the wrong place · very small | You did not multiply the 0~1 ratios by the original width · height — section 5 |
| Scores strangely low (around 0.7) | You applied a sigmoid again to scores that are already 0~1 — use them as they are |
| A truck · a car overlap in the same place | You skipped the overlap cleanup (IoU 0.7) — section 7 |
| All joints bunched up in the top-left of the photo | You skipped the inverse transform (used the cropped-photo coordinates as is) — section 9 |
| No joints found at all | You did ÷255 for the pose model — this model takes 0~255 as is |

## Summary
- Preprocessing: stretch to 640 x 640 (the way the model learned) → RGB → ÷255 → NCHW.
- Output: 300 candidate slots — each slot has a box (0~1 ratios) + scores for 80 classes (already 0~1). One slot per object.
- Postprocessing: score filter (0.5) → corner coordinates × original size → tidy up only heavy overlaps (IoU 0.7).
- Joints: person box → crop at 4:3 · 1.25x → highest bin score ÷ 2 → inverse transform.
- Time: the faster inference gets, the bigger the share of pre- and post-processing.

## Things to try
1. Lower it with `decode(..., conf_th=0.3)`, and also try `iou_th=1.0` (no cleanup). How does the number of boxes change?
2. What happens if you feed it with letterbox (keep the ratio and fill with gray)? Check whether the scores drop when you feed it differently from how the model learned.
3. Change `DEVICE` to `"CPU"` · `"NPU"` and compare the table in section 8.
4. Check that your results match the library.
   <details><summary>Hint</summary>

   `from easydetect import Detector; r = Detector(str(ED), device="CPU")(frame, conf=0.5)[0]; print(r.boxes.xyxy, r.boxes.cls, r.boxes.conf)`
   </details>
5. Count only "person" in the webcam video and show the number on the screen (`cls_k == 0`).

---
# 🛠 Mini project — People counter monitor

**What you build**: Count **only people (class 0)** in the webcam video, show the count on the screen, and log the time · number of people to a **CSV** file every second.
When it ends, it tells you the busiest moment and the average. (A very small version of measuring how crowded a store · classroom is)

**What you use**: `to_blob` · `decode` (this notebook — exactly what we built with no library), `live`

In [ ]:
from IPython.display import Image as IPImage

def live(process, seconds=15, index=0, width=640):
    """Show webcam video in the notebook for `seconds` seconds, changing each frame with process(photo) → photo. Press I I to stop."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    still = None if cap.isOpened() and cap.read()[0] else cv2.imread(PHOTO)   # if there is no webcam, use the photo
    handle = display(None, display_id=True)
    t0, n, out = time.time(), 0, None
    try:
        while time.time() - t0 < seconds and (still is None or n < 10):
            if still is None:
                ok, f = cap.read()
                if not ok:
                    break
            else:
                f = still.copy()
            out = process(f)
            s = width / out.shape[1]
            small = cv2.resize(out, (width, int(out.shape[0] * s))) if s < 1 else out
            handle.update(IPImage(data=cv2.imencode(".jpg", small)[1].tobytes()))
            n += 1
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
    print(f"{n} frames · {n / max(time.time() - t0, 1e-6):.1f} FPS" + (" (no webcam, so faking it with the photo)" if still is not None else ""))
    return out

In [ ]:
import csv
from datetime import datetime

log, last_log = [], [0.0]

def count_people(f):
    r = compiled(to_blob(f))
    xyxy, c, s, _ = decode(r[compiled.output("boxes")][0], r[compiled.output("scores")][0], f.shape)
    people = xyxy[c == 0]                                           # people only
    for x1, y1, x2, y2 in people.astype(int):
        cv2.rectangle(f, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(f, f"people: {len(people)}", (10, 36), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 0, 255), 3)
    if time.time() - last_log[0] >= 1.0:                            # log once per second
        log.append((datetime.now().strftime("%H:%M:%S"), len(people)))
        last_log[0] = time.time()
    return f

live(count_people, seconds=20)
with open("people_log.csv", "w", newline="", encoding="utf-8") as fp:
    csv.writer(fp).writerows([("time", "people")] + log)
if log:
    counts = [n for _, n in log]
    peak_ = max(log, key=lambda x: x[1])
    print(f"{len(log)} records → people_log.csv · average {np.mean(counts):.1f} people · max {peak_[1]} people ({peak_[0]})")

**Try more**
- Count a different class instead of people (e.g. bottle 39, cup 41 — look up the number in `NAMES`)
- Count the people in the left half / right half of the screen separately (split by the box's center x)
- Log only the moment the count changes from 0 people → 1 or more (an entry alert)